# 手写数字识别 — Notebook 入口

这个 notebook 是本项目的**入口**：用 `src/` 里的代码完成训练、评估与预测导出。

## 你将完成什么
1. 安装依赖（只需一次）
2. 在 MNIST 上训练并将产物写入 `outputs/`
3. 评估（混淆矩阵 + 误分类样例图）
4. （可选）对无标签图片文件夹导出预测 CSV

## 你应该重点查看的产物
- `outputs/checkpoints/best_model.pt`
- `outputs/logs/history.json`
- `outputs/logs/run_manifest.json`
- `outputs/figures/training_curves.png`
- `outputs/figures/confusion_matrix.png`
- `outputs/figures/misclassified_grid.png`


## 0) 环境检查
运行下一格代码，确认你在正确的仓库目录与 Python 环境中。

In [ ]:
import sys
from pathlib import Path

print('Python:', sys.version)
print('Executable:', sys.executable)
print('CWD:', Path.cwd())
print('Repo contains src?:', (Path.cwd() / 'src').is_dir())


## 1) 安装依赖（只需一次）
如果你已经在当前环境安装过依赖，可以跳过。

In [ ]:
# 如果在 Jupyter 里安装失败，请改用终端执行：
# python -m pip install -r requirements.txt

import sys
import subprocess

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


## 2) 训练 MNIST（推荐先跑通基线）
这一部分实际调用的是 `python -m src.train`。

建议先用较小的 epochs（例如 3）验证整条流水线可用，然后再拉长训练做对比实验。

In [ ]:
import sys
import subprocess
from pathlib import Path

project_root = Path('.').resolve()
epochs = 3
batch_size = 64
learning_rate = 1e-3
seed = 42

cmd = [
    sys.executable, '-m', 'src.train',
    '--dataset-name', 'mnist',
    '--project-root', str(project_root),
    '--epochs', str(epochs),
    '--batch-size', str(batch_size),
    '--learning-rate', str(learning_rate),
    '--seed', str(seed),
]

print('运行命令:', ' '.join(cmd))
subprocess.check_call(cmd)


## 3) 检查训练产物
确认训练完成后关键文件都已生成。

In [ ]:
import json
from pathlib import Path

outputs_dir = Path('outputs')
checkpoint = outputs_dir / 'checkpoints' / 'best_model.pt'
history_path = outputs_dir / 'logs' / 'history.json'
manifest_path = outputs_dir / 'logs' / 'run_manifest.json'

for path in [checkpoint, history_path, manifest_path]:
    print(path, '是否存在?', path.exists())

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('run_manifest 内容:', manifest)


## 4) 评估（混淆矩阵 + 误分类样例图）
这一部分调用的是 `python -m src.evaluate`，并将图像产物写入 `outputs/figures/`。

In [ ]:
import sys
import subprocess
from pathlib import Path

checkpoint = Path('outputs/checkpoints/best_model.pt').resolve()

cmd = [
    sys.executable, '-m', 'src.evaluate',
    '--checkpoint', str(checkpoint),
    '--dataset-name', 'mnist',
    '--project-root', str(Path('.').resolve()),
]

print('运行命令:', ' '.join(cmd))
subprocess.check_call(cmd)


## 5) 查看图像产物
如果你的 Jupyter 环境支持图片展示，可以在文件浏览器里打开这些 PNG；此处也会检查文件是否存在。

In [ ]:
from pathlib import Path

fig_dir = Path('outputs/figures')
paths = [
    fig_dir / 'training_curves.png',
    fig_dir / 'confusion_matrix.png',
    fig_dir / 'misclassified_grid.png',
]

for p in paths:
    print(p, 'exists?', p.exists())


## 6) （可选）对无标签图片导出预测结果
把无标签图片放到一个文件夹里（例如 `sample_predict_images/`），然后运行下一格。

预期输出：`outputs/predictions/predictions.csv`

In [ ]:
import sys
import subprocess
from pathlib import Path

image_dir = Path('sample_predict_images')
if not image_dir.exists():
    raise FileNotFoundError('请创建 sample_predict_images/ 并放入无标签图片后再运行此单元格。')

checkpoint = Path('outputs/checkpoints/best_model.pt').resolve()
cmd = [
    sys.executable, '-m', 'src.predict',
    '--checkpoint', str(checkpoint),
    '--image-dir', str(image_dir.resolve()),
    '--project-root', str(Path('.').resolve()),
]

print('运行命令:', ' '.join(cmd))
subprocess.check_call(cmd)

print('CSV 已生成:', Path('outputs/predictions/predictions.csv').resolve())
